# HRH: full pipeline

Runtime → Change runtime type → **GPU**. Everything is written to Google Drive, so an interrupted
cell can simply be rerun: finished subjects and models are skipped.

1. **Development run**: brain RDMs, features for all models, evaluation.
2. **Law**: brain-free proxies, fit and freeze the law, predictions for the held-out models.
3. **Registration**: the predictions are registered (OSF) before any held-out brain analysis.
4. **Confirmatory test**: evaluation including the held-out models.

In [ ]:
# Settings
REPO = "https://github.com/LegallyCoder/hrh"
OUT  = "/content/drive/MyDrive/hrh_results"   # persistent

from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Code + dependencies (~1 min)
%cd /content
!rm -rf hrh && git clone -q $REPO hrh
%cd /content/hrh
!pip -q install -e ".[models,dev]"
!pytest -q
!git log --oneline -1

## 1. Development run
Brain and development evaluation are already on Drive if you ran them before; the features cell skips finished models, so it only extracts new ones.

In [ ]:
!python run_hrh.py brain --out $OUT                 # ~1 h: EEG download, RDMs, ceilings

In [ ]:
!python run_hrh.py features --out $OUT --clean-cache  # all models incl. held-out (Grams only, no brain data)

In [ ]:
!python run_hrh.py eval --out $OUT --paper $OUT/paper  # held-out models are excluded until registered

## 2. Law (brain-free proxies → fit → held-out predictions)
`proxies` uses only Gram matrices, the images and human category labels. `law` is fit on the
development models. `predict` writes `registration/predictions_heldout.json` and its SHA-256.

In [ ]:
!python run_hrh.py proxies --out $OUT
!python run_hrh.py law     --out $OUT
!python run_hrh.py predict --out $OUT --reg-dir $OUT/registration

In [ ]:
from google.colab import files
files.download(f"{OUT}/registration/predictions_heldout.json")
files.download(f"{OUT}/registration/predictions_heldout.sha256")

## 3. Registration (the one manual step, ~5 min)
Upload `predictions_heldout.json` to a new OSF registration (osf.io → Create project → upload →
Registrations → register). Note the SHA-256 printed above. Then continue below in this same
session; the final evaluation reads the registration from Drive and checks its hash.

## 4. Confirmatory test

In [ ]:
import os
os.environ["HRH_REG_DIR"] = f"{OUT}/registration"     # registered predictions + hash
!python run_hrh.py eval --out $OUT --paper $OUT/paper   # includes the held-out law_test

In [ ]:
# Package the results (a few MB)
!cd $OUT && zip -qr /content/hrh_results.zip paper/generated paper/figures summary_*.json models_*.csv proxies_*.json law.json
from google.colab import files
files.download('/content/hrh_results.zip')

## 5. Model-to-model convergence (CPU, ~15 min)
PRH's own metric between all pairs of models, by depth. No GPU needed.

In [ ]:
!python run_hrh.py convergence --out $OUT
from google.colab import files
files.download(f"{OUT}/convergence_grid4.json")
files.download(f"{OUT}/A_matrices_grid4.npz")

## 6. Controlled objective experiment (GPU)
One architecture (ViT-Tiny/8, 64 px), one dataset (ImageNet-1k at 64x64), one budget; only the
objective changes: supervised, MAE, JEPA, data2vec, two seeds each. Checkpoints are saved to Drive
every 500 steps, so a disconnect loses at most a few minutes: rerun the same cell to resume.

**6a. Data** (once per session, ~20-30 min): downloads 1.7 GB and decodes it to local disk.

In [ ]:
!python -c "from hrh.train import prepare_data as p; p('/content/imagenet64'); p('/content/imagenet64', 'validation')"

**6b. Smoke test** (~3 min): 30 steps of every objective. If this fails, stop and send the error.

In [ ]:
!python run_hrh.py train --objective all --seeds 0 --max-steps 30 --ckpt-dir /content/smoke_ckpt

**6c. Training** (8 runs, ~30-45 min each on a T4). Resumes automatically after a disconnect.

In [ ]:
!python run_hrh.py train --objective all --seeds 0,1 --ckpt-dir $OUT/checkpoints --knn

**6d. Brain evaluation** of the trained networks (features, registered test, proxies).

In [ ]:
import os
os.environ["HRH_CKPT_DIR"] = f"{OUT}/checkpoints"
!python run_hrh.py features --out $OUT --families ctl-sup,ctl-mae,ctl-jepa,ctl-data2vec
!python run_hrh.py eval --out $OUT --paper $OUT/paper --pools grid4
!python run_hrh.py proxies --out $OUT
from google.colab import files
for f in ("summary_grid4.json", "models_grid4.csv", "proxies_grid4.json"):
    files.download(f"{OUT}/{f}")